# Sabaic OCR YOLO — Google Colab
Custom YOLO, loss, NMS and synthetic generator are implemented inside this repository. No Ultralytics/TRDG or external pretrained weights are used. Set Colab Runtime -> Change runtime type -> GPU before full training.

In [ ]:
!git clone https://github.com/7eaur/Sabaic-OCR-YOLO.git
%cd Sabaic-OCR-YOLO
!pip install -r requirements.txt
!pip install -e . --no-deps --no-build-isolation
!python scripts/check_environment.py
!pytest -q

## Persistent checkpoints on Google Drive
This keeps `best.pt`, `last.pt` and epoch checkpoints after Colab disconnects.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
drive_ckpt = Path('/content/drive/MyDrive/Sabaic-OCR-YOLO/checkpoints')
drive_ckpt.mkdir(parents=True, exist_ok=True)
local_ckpt = Path('checkpoints')
if local_ckpt.is_symlink():
    local_ckpt.unlink()
elif local_ckpt.exists():
    shutil.rmtree(local_ckpt)
local_ckpt.symlink_to(drive_ckpt, target_is_directory=True)
print('Persistent checkpoints:', local_ckpt.resolve())

## 1) Add and validate the font
Upload `NotoSansOldSouthArabian-Regular.ttf` supplied for the project. The font is kept local and is not committed by this notebook.

In [ ]:
from google.colab import files
from pathlib import Path
Path('assets/fonts').mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    if name.lower().endswith('.ttf'):
        Path('assets/fonts/NotoSansOldSouthArabian-Regular.ttf').write_bytes(data)
!python scripts/validate_font.py

## 2) Generate/resume the reviewed synthetic baseline
Target: 5,000 train + 500 validation + 100 synthetic test images. `--resume` safely continues an interrupted generation.

In [ ]:
!python scripts/generate_synthetic.py --train 5000 --val 500 --test 100 --resume

## 3) Mandatory synthetic stage gate
Training is not started unless counts, labels, transcripts, all 30 classes and cross-split leakage checks pass.

In [ ]:
!python scripts/review_synthetic_stage.py --require-train 5000 --require-val 500 --require-test 100
!python scripts/fit_anchors.py --images data/synthetic/images/train --labels data/synthetic/labels/train
!python scripts/sanity_overfit.py

## 4) Full synthetic pretraining on GPU
The reviewed anchors are already stored in `config/model.json`. This run produces `checkpoints/synthetic/best.pt` and `last.pt`.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU runtime required for the full training run.'
!python scripts/train_synthetic.py --config config/train_synthetic.json

### Resume after a disconnect
Set `resume` in `config/train_synthetic.json` to `checkpoints/synthetic/last.pt`, then rerun the training cell. Do not call a smoke/preflight value final accuracy.

## 5) Real dataset gate and fine-tuning
The fine-tuning Train split must contain at least **200 real labeled images**. Synthetic images never count toward this minimum. Validation and test are kept separate.

In [ ]:
!python scripts/audit_real_dataset.py --min-train 200 --min-val 20 --min-test 20
!python scripts/validate_labels.py --images data/real/images/train --labels data/real/labels/train --preview-dir outputs/real_label_previews
!python scripts/finetune_real.py --config config/train_real.json

## 6) Final evaluation on independent Real Test
Reports detection metrics plus character/word correct-wrong counts, CER and WER.

In [ ]:
!python scripts/evaluate.py --checkpoint checkpoints/real_finetune/best.pt

## 7) Inference

In [ ]:
# Example after uploading test.jpg
!python scripts/infer.py --checkpoint checkpoints/real_finetune/best.pt --image test.jpg